In [ ]:
import plotly.express as px
import torch

from aare.constants import ANYTIME
from aare.constants import TIME, TEMP
from aare_train.params import read_params
from aare_train.preparation import resample, prepare_ts_aare_temp, interpolate_aare_temp, remove_outliers
from aare_influx.remote_existenz_store import RemoteExistenzStore
from aare.pd_utils import fill_with_hard_limit
from aare_train.darts_utils import to_ts

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
torch.set_float32_matmul_precision("medium")

In [ ]:
# logging.basicConfig(level="DEBUG")

In [ ]:
params = read_params()
store = RemoteExistenzStore()

# Deeper Questions on covariates

After the analysis of the other variables and talking to people, some questions need to be answered.

- Is sunshine duration (smn/ss) a good proxy for global radiation (smn/rad), because there is no radiation forecast?
- Is precipitation (smn/rr) a good proxy for relative humidity (smn/rh), because there is no rh forecast?
- Is the water temperature of the thun lake slow/static enough that it could be used to aid forecasting as past cov (no lake temp forecast)? Spiez (2093) is probably the closest you get to the river temp, but it has no temperature :( Can try Interlaken (2457), which is the river connection between Brienzersee and Thunersee.
- Imputation/Missing Data
  - How many gaps does the air temperature have?
  - Can the air temp be filled the same way as the water temp?
  - THIS SHOULD BE DONE FOR ALL VARIABLES THAT ARE INCLUDED IN MODEL TRAINING


In [ ]:
df = store.query(
    ANYTIME,
    [
        "hydro/temperature:mean_1h@bern",
        "hydro/temperature:mean_1h@thun",
        "hydro/temperature:mean_1h@int",
        "hydro/flow:mean_1h@bern",
        "hydro/flow:mean_1h@thun",
        "smn/tt:mean_1h@bern",
        "smn/tt:mean_1h@thun",
        "smn/tt:mean_1h@int",
        "smn/ss:sum_1h@bern",
        "smn/rad:sum_1h@bern",
        "smn/rr:sum_1h@bern",
        "smn/rh:sum_1h@bern",
    ],
)
df

In [ ]:
df = resample(df)

In [ ]:
df.describe().T

In [ ]:
# As a reminder, these are the gaps that are still there after getting the most out of temp_bern
temp_bern = prepare_ts_aare_temp(df[[TIME, "temperature_bern"]].rename({"temperature_bern": TEMP}, axis="columns"))
temp_bern.gaps().sort_values("gap_size", ascending=False)

## Sunshine Duration as Radiation Proxy

In [ ]:
df_s = df.copy()
df_s[["rad_bern", "ss_bern"]] /= df[["rad_bern", "ss_bern"]].max()
px.scatter(df_s, x=TIME, y=["rad_bern", "ss_bern"]).update_traces(marker={"size": 2})

Visually, there clearly is a high correlation, but it seems that some fine patterns are different like

- some days with low but non-zero sunshine duration have flat 0 radidation
- radiation is often maxed without fluctuations as soon as the sunshine duration approaches ~30min per hour, while sunshine duration has more of a curve most days

In [ ]:
rad_bern, ss_bern = to_ts(df, col="rad_bern"), to_ts(df, col="ss_bern")

In [ ]:
rad_bern.gaps().sort_values("gap_size", ascending=False)

In [ ]:
ss_bern.gaps().sort_values("gap_size", ascending=False)

In [ ]:
df_f = interpolate_aare_temp(df[[TIME, "ss_bern"]], columns="ss_bern")
df_f

In [ ]:
px.scatter(df_f, x=TIME, y="ss_bern", color="filled")

Looking at the imputation with linear and cubic interpolation, it clearly seems to be the wrong tool. I think a moving median would do be much better.

In [ ]:
df_f = interpolate_aare_temp(df[[TIME, "ss_bern"]], columns="ss_bern")
df_f

In [ ]:
df_i = fill_with_hard_limit(
    df[[TIME, "ss_bern"]],
    limit=10,
    fill_func="median",
    columns=["ss_bern"],
    add_was_filled=True,
)
df_i

In [ ]:
df_i["original"] = df["ss_bern"]

In [ ]:
df_i

In [ ]:
df[[TIME, "ss_bern"]]

In [ ]:
df["ss_bern"]

In [ ]:
px.scatter(df_i, x=TIME, y=["ss_bern"], color="ss_bern_filled")

In [ ]:
px.scatter(df_i, x=TIME, y=["ss_bern", "original"])

In [ ]:
to_ts(df_i, col="ss_bern").gaps().sort_values("gap_size", ascending=False)

## Precipitation as Relative Humidity Proxy

In [ ]:
px.scatter(df, x=TIME, y=["rh_bern", "rr_bern"]).update_traces(marker={"size": 2})

## Water temperature BERN, THUN and INT

In [ ]:
px.scatter(df, x=TIME, y=["temperature_bern", "temperature_thun", "temperature_int"]).update_traces(marker={"size": 2})

Starts:

- BERN: 2001
- THUN: 2003
- INT: 2018

INT is much lower (as expected), but not suitable as baseline to just predict the diff to bern on top of (it has even higher variance than bern and thun).

## Flow Bern

In [ ]:
px.scatter(df, x=TIME, y="flow_bern").update_traces(marker={"size": 2})

In [ ]:
to_ts(df, col="flow_bern").gaps()[["gap_size"]].value_counts()

In [ ]:
df["flow_bern_diff"] = df["flow_bern"].diff()
df["flow_bern_diff_abs"] = df["flow_bern_diff"].abs()
df[[TIME, "flow_bern", "flow_bern_diff", "flow_bern_diff_abs"]].describe().T

In [ ]:
outlier_quantile = 0.999
print("abs diff threshold:", df["flow_bern_diff_abs"].quantile(outlier_quantile))
print("low flow threshold:", df["flow_bern"].quantile(1 - outlier_quantile))
print("high flow threshold:", df["flow_bern"].quantile(outlier_quantile))

From looking at the data, I would adjust the limits as follows:

- low flow: 30 (33 cuts off data that looks legit)
- high flow: no outliers at the top. the 560 was a legit flood.
- diff: 14 (in the flood of summer 2021, the peak was 14.05 it seems.
  Nevermind, there were two legit instances with ~40, in fall 2011 there's ~45.
  maybe diff to the median might be better? or second derivative? rolling_std?)

In [ ]:
df_f = df.copy()

In [ ]:
df_f["flow_bern_diff"] = df_f["flow_bern"].diff()
df_f["flow_bern_diff_abs"] = df_f["flow_bern_diff"].abs()
df_f[[TIME, "flow_bern", "flow_bern_diff", "flow_bern_diff_abs"]].describe().T

In [ ]:
df_f["flow_bern_diff_abs_std"] = df_f["flow_bern_diff_abs"].rolling(5).std()
df_f["flow_bern_diff_abs_median"] = df_f["flow_bern_diff_abs"].rolling(5).median()
df_f["flow_bern_median"] = df_f["flow_bern"].rolling(3).median()
df_f["flow_bern_diff_abs_std_error"] = df_f["flow_bern_diff_abs"] - df_f["flow_bern_diff_abs"].mean()
df_f["flow_bern_diff_abs_median_error"] = (df_f["flow_bern_diff_abs"] - df_f["flow_bern_diff_abs_median"]).abs()
df_f["flow_bern_median_error"] = (df_f["flow_bern"] - df_f["flow_bern_median"]).abs()

In [ ]:
px.scatter(df_f, x=TIME, y="flow_bern", color=df_f["flow_bern_diff_abs_std_error"] > df_f["flow_bern_diff_abs_std"] * 3)

In [ ]:
px.scatter(df_f, x=TIME, y="flow_bern", color=df_f["flow_bern_diff_abs_median_error"] > 10)

In [ ]:
px.scatter(df_f, x=TIME, y="flow_bern", color=df_f["flow_bern_median_error"] > 18)

In [ ]:
df_f2 = remove_outliers(df_f, 30, 99999, 14.5, col="flow_bern")
df_f["flow_bern_clean"] = df_f2["flow_bern"]

In [ ]:
df_f["flow_bern_r48"] = df_f["flow_bern_clean"].rolling(48, center=True, min_periods=10).median()
df_f["flow_bern_error_to_r48"] = (df_f["flow_bern_clean"] - df_f["flow_bern_r48"]).abs()
# df_f["flow_bern_error_to_r48_rolling_std"] = df_f["flow_bern_error_to_r48"].rolling(24, center=True, min_periods=5).std()
df_f["bad_boy"] = (
    df_f["flow_bern_error_to_r48"] - df_f["flow_bern_error_to_r48"].mean() > df_f["flow_bern_error_to_r48"].std() * 2
)  # df_f["flow_bern_error_to_r48_rolling_std"] * 2

In [ ]:
px.scatter(df_f, x=TIME, y=["flow_bern", "flow_bern_clean", "flow_thun", "flow_bern_r48"]).update_traces(
    marker={"size": 2}
)

In [ ]:
df_f["flow_bern_thun"] = df_f["flow_bern"] - df_f["flow_thun"]
px.scatter(df_f, x=TIME, y=["flow_bern", "flow_bern_clean", "flow_thun", "flow_bern_thun"]).update_traces(
    marker={"size": 2}, visible="legendonly"
)

In [ ]:
px.scatter(df_f, x=TIME, y="flow_bern_clean", color="bad_boy").update_traces(marker={"size": 2})

In [ ]:
px.scatter(df_i, x=TIME, y="flow_bern_clean", color="filled").update_traces(marker={"size": 2})

In [ ]:
from aare_train.features.flow_bern import FlowBern
from aare_train.preparation import interpolate_continuous

df_i = FlowBern()._remove_faulty_periods(df)
df_i = remove_outliers(df_i, 30, 99999, 14.5, col="flow_bern")
df_i = interpolate_continuous(df_i, 3, 23, drop_filled=False, columns="flow_bern")
df_i["flow_bern_clean"] = df_i["flow_bern"]
df_i["flow_bern"] = df["flow_bern"]

In [ ]:
px.scatter(df_i, x=TIME, y="flow_bern_clean", color="filled").update_traces(marker={"size": 2})

In [ ]:
to_ts(df_i, col="flow_bern_clean").gaps().sort_values("gap_size", ascending=False)

In [ ]:
df_x = FlowBern()._remove_faulty_periods(df)
px.scatter(df_x, x=TIME, y="flow_bern").update_traces(marker={"size": 2})

In [ ]:
px.scatter(df_i, x=TIME, y=["flow_bern_clean", "flow_thun"]).update_traces(marker={"size": 2})

## Air temperature

In [ ]:
px.scatter(df, x=TIME, y=["tt_bern", "tt_thun", "tt_int"]).update_traces(marker={"size": 2})

In [ ]:
tt_bern = to_ts(df, col="tt_bern")

In [ ]:
tt_bern.gaps().sort_values("gap_size", ascending=False)

### Imputation of air temperature

In [ ]:
df_f = interpolate_aare_temp(df[[TIME, "tt_bern"]], columns="tt_bern")
df_f

In [ ]:
px.scatter(df_f, x=TIME, y="tt_bern", color="filled")